In [1]:
!pip install transformers datasets scikit-learn pandas pyarrow

In [2]:
import pandas as pd
import numpy as np
import pickle

from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [3]:
# load the dataset
dataset = load_dataset("ButterChicken98/plantvillage-image-text-pairs")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
# convert the dataset to pandas dataframe
df = dataset["train"].to_pandas()
df.head()

,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [5]:
# check the column names in the dataset
print(df.columns)

Index(['image', 'caption', 'captions'], dtype='object')


In [7]:
# convert caption lists into individual rows and prepare text dataset
df = df.explode("captions")
df = df.rename(columns={
    "captions": "text",
    "caption": "label_name"
})
df = df[["text", "label_name"]]
df.head()

,text,label_name
0,A vibrant green and healthy tomato leaf with s...,Tomato healthy
0,"A healthy Solanum lycopersicum leaf, free of d...",Tomato healthy
0,"A fresh tomato leaf outdoors, glowing in sunli...",Tomato healthy
0,"A clean and healthy tomato leaf image, perfect...",Tomato healthy
1,A tomato leaf showing dark brown lesions and w...,Tomato Late blight


In [8]:
df = df.rename(columns={"caption": "label_name"})
df = df[["text", "label_name"]]
df.head()

,text,label_name
0,A vibrant green and healthy tomato leaf with s...,Tomato healthy
0,"A healthy Solanum lycopersicum leaf, free of d...",Tomato healthy
0,"A fresh tomato leaf outdoors, glowing in sunli...",Tomato healthy
0,"A clean and healthy tomato leaf image, perfect...",Tomato healthy
1,A tomato leaf showing dark brown lesions and w...,Tomato Late blight


In [9]:
# convert disease names into numerical labels
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label_name"])
num_classes = len(encoder.classes_)
print("Number of classes:", num_classes)

Number of classes: 15


In [10]:
# split the dataset into training and testing sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(len(train_df), len(test_df))

66041 16511


In [11]:
# convert pandas dataframes into HuggingFace dataset format
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

In [12]:
# load the DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [14]:
tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [15]:
# remove unused columns and format the dataset for PyTorch
tokenized_train = tokenized_train.remove_columns(["text", "label_name"])
tokenized_test = tokenized_test.remove_columns(["text", "label_name"])

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

In [16]:
# load the DistilBERT model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
# define training parameters for DistilBERT
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)

In [18]:
# define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [19]:
# initialize the trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

In [20]:
# train the DistilBERT model
trainer.train()

Step,Training Loss
500,0.511642
1000,0.006485
1500,0.002263
2000,0.001145
2500,0.000693
3000,0.000438
3500,0.000314
4000,0.000223
4500,0.000163
5000,0.000123


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12384, training_loss=0.021154734034890498, metrics={'train_runtime': 2487.7636, 'train_samples_per_second': 79.639, 'train_steps_per_second': 4.978, 'total_flos': 6562730734145280.0, 'train_loss': 0.021154734034890498, 'epoch': 3.0})

In [21]:
# evaluate the model
trainer.evaluate()

{'eval_loss': 3.8929392758291215e-06,
 'eval_accuracy': 1.0,
 'eval_f1': 1.0,
 'eval_runtime': 65.3544,
 'eval_samples_per_second': 252.638,
 'eval_steps_per_second': 15.791,
 'epoch': 3.0}

In [22]:
# save the trained model
model.save_pretrained("plant_disease_text_model")
tokenizer.save_pretrained("plant_disease_text_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('plant_disease_text_model/tokenizer_config.json',
 'plant_disease_text_model/tokenizer.json')

In [23]:
# save the label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

In [24]:
# test the saved model
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="plant_disease_text_model",
    tokenizer="plant_disease_text_model"
)

text = "Tomato leaf with yellow spots and curling edges"
prediction = classifier(text)

print(prediction)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_12', 'score': 0.9994438290596008}]


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
# Copy trained model to Google Drive
!cp -r plant_disease_text_model "/content/drive/My Drive/INTERNSHIP/"
!cp label_encoder.pkl "/content/drive/My Drive/INTERNSHIP/"